In [1]:
import os
from pathlib import Path
backend_path = Path('/home/guyder/web-path-tracing-renderer/backend')
os.chdir(backend_path)

In [2]:
import numpy as np
from scipy.spatial.transform import Rotation as R_scipy

def decompose_transform(transform):
    matrix = np.array(transform.matrix)
    
    # Translation
    translation = matrix[0:3, 3]
    
    # Linear part 3x3
    linear_part = matrix[0:3, 0:3]
    
    # Scale
    scale = np.array([
        np.linalg.norm(linear_part[:, 0]),
        np.linalg.norm(linear_part[:, 1]),
        np.linalg.norm(linear_part[:, 2])
    ])
    
    # Rotation matrix
    rotation_matrix = np.zeros((3, 3))
    for i in range(3):
        if scale[i] > 1e-8:
            rotation_matrix[:, i] = linear_part[:, i] / scale[i]
        else:
            rotation_matrix[:, i] = linear_part[:, i]
    
    # Handle negative scale
    if np.linalg.det(rotation_matrix) < 0:
        scale[0] *= -1
        rotation_matrix[:, 0] *= -1
    
    # Euler angles
    rot_scipy = R_scipy.from_matrix(rotation_matrix)
    euler_angles = rot_scipy.as_euler('xyz', degrees=True)
    
    return translation, scale, euler_angles

import numpy as np

def decompose_sensor(trafo):
    # Получаем 4x4 матрицу из объекта Transform
    matrix = trafo.matrix
    
    # 1. Позиция (Position) - это 4-й столбец (индекс 3)
    # В Mitsuba столбцы доступны через обращение к матрице
    origin = mi.Point3f(matrix[0, 3], matrix[1, 3], matrix[2, 3])
    
    # 2. Направление взгляда (Forward / Look Direction) 
    # Это 3-й столбец (индекс 2), так как камера смотрит в +Z
    forward = mi.Vector3f(matrix[0, 2], matrix[1, 2], matrix[2, 2])
    
    # 3. Вектор "Верх" (Up)
    # Это 2-й столбец (индекс 1), так как локальный верх - это +Y
    up = mi.Vector3f(matrix[0, 1], matrix[1, 1], matrix[2, 1])
    
    # 4. Точка LookAt (Target)
    # Находим любую точку на луче взгляда. 
    # Обычно берут origin + forward
    target = origin + forward
    
    return origin, target, up

In [19]:
def atx(x, y, z): # a - arr, t - to, x - xyz
    return {
        'x': x,
        'y': y,
        'z': z,
    }

def ath(x, y, z): # a - arr, t - to, h - hex
    x = int(x*255)
    y = int(y*255)
    z = int(z*255)
    return f"#{x:02x}{y:02x}{z:02x}"

In [37]:
import mitsuba as mi
mi.set_variant('llvm_ad_rgb')
cb = mi.cornell_box()

print(decompose_sensor(cb['sensor']['to_world']))

objects = ['light', 'floor', 'ceiling', 'back', 'green-wall', 'red-wall', 'small-box', 'large-box']
mat = {}

scene = {
    'objects': [],
    'sensor': {
        'position': atx(0, 0, 3.9),
        'lookAt': atx(0, 0, 2.9),
        'up': atx(0, 1, 0),
        'fov': 39.3077,
    },
    'width': 500,
    'height': 500,
    'spp': 256,
}

for i, obj in enumerate(objects):
    mat = cb[obj]['to_world']
    bsdf = cb[obj]['bsdf']['id']
    tr, sc, an = decompose_transform(mat)
    scene['objects'].append({
        'id': obj,
        'type': cb[obj]['type'],
        'position': atx(*map(float, tr)),
        'scale': atx(*map(float, sc)),
        'rotation': atx(*map(float, an)),
        'color': ath(*map(float, cb[bsdf]['reflectance']['value'])),
        'material': 'diffuse',
    })
    print(f'\nobj: {obj}')
    print(f'{'transform':<10} : {tr}')
    print(f'{'scale':<10} : {sc}')
    print(f'{'rotate':<10} : {an}')

([[0, 0, 3.9]], [[0, 0, 2.9]], [[0, 1, 0]])

obj: light
transform  : [0.   0.99 0.01]
scale      : [0.22999999 0.19       0.19      ]
rotate     : [90.0000025  0.         0.       ]

obj: floor
transform  : [ 0. -1.  0.]
scale      : [0.99999994 1.         1.        ]
rotate     : [-90.0000025   0.          0.       ]

obj: ceiling
transform  : [0. 1. 0.]
scale      : [0.99999994 1.         1.        ]
rotate     : [90.0000025  0.         0.       ]

obj: back
transform  : [ 0.  0. -1.]
scale      : [1. 1. 1.]
rotate     : [0. 0. 0.]

obj: green-wall
transform  : [1. 0. 0.]
scale      : [1.         0.99999994 1.        ]
rotate     : [  0.        -89.9999975   0.       ]

obj: red-wall
transform  : [-1.  0.  0.]
scale      : [1.         0.99999994 1.        ]
rotate     : [-0.        89.9999975  0.       ]

obj: small-box
transform  : [ 0.335 -0.7    0.38 ]
scale      : [0.30000004 0.3        0.30000004]
rotate     : [  0.        -16.9999977   0.       ]

obj: large-box
transform  : [-

/tmp/ipykernel_52932/2285507808.py:35: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  euler_angles = rot_scipy.as_euler('xyz', degrees=True)


In [38]:
scene['objects'][0]['material'] = 'emitter'

In [39]:
from routers.render import test_render

In [40]:
import mitsuba as mi
import matplotlib.pyplot as plt

img = test_render(scene)

mi.util.write_bitmap('/home/guyder/web-path-tracing-renderer/notebooks/cornell_box.png', img)

In [36]:
import json
with open('/home/guyder/web-path-tracing-renderer/notebooks/cornell_box.json', 'w', encoding='utf-8') as f:
    json.dump(scene, f, ensure_ascii=False, indent=4)

In [ ]:
cb['light']

In [43]:
result = []
spath = Path('/home/guyder/web-path-tracing-renderer/backend/scenes')

for shash in spath.iterdir():
    sname = list(shash.glob('*.json'))[0]
    result.append({
        'dir_name': shash.name,
        '.txt name': sname.stem,
    })

result

[{'dir_name': 'contell_box', '.txt name': 'data'}]